In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

from langgraph.graph import START, END, StateGraph
# llm= init_chat_model(
#     model="qwen3.6", 
#     model_provider="ollama",
#     temperature=0.3
# )

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.7
)


class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)
builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)
graph = builder.compile()




In [5]:
message = {"role": "user", "content": "who walked on the moon first? print only name"}
response = graph.invoke({"messages": [message]})

response["messages"]

[HumanMessage(content='who walked on the moon first? print only name', additional_kwargs={}, response_metadata={}, id='221ba268-0ded-4977-9031-2c7a3c550d3a'),
 AIMessage(content='Neil Armstrong', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 45, 'total_tokens': 48, 'completion_time': 0.008204584, 'completion_tokens_details': None, 'prompt_time': 0.00223694, 'prompt_tokens_details': None, 'queue_time': 0.232280579, 'total_time': 0.010441524}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f5bb1-10f4-71e1-b089-3d64e4a3307a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 3, 'total_tokens': 48})]

In [ ]:
state = None
while True:
    input_message = input("User: ")
    if input_message.lower() in ["exit", "quit"]:
        break
    if state is None:
        state : State = {
                          "messages": [{"role": "user", "content": input_message}]
                         }
    else:
        state["messages"].append({"role": "user", "content": input_message})
    
    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)    